# [교육] 온라인 강의 사용자 로그 분석 및 시각화
- 사용자 로그로 온라인 강의의 ‘참여’와 ‘이탈’을 어떻게 읽을 것인가?

- 온라인 강의 데이터 기반 대시보드 구축 프로젝트
    - 학습자 참여도 진단 → 이탈 구간 발견 → 개선 방향 제시

### 주제 목표
- 사용자 로그 기반 온라인 강의 학생참여도 진단 및 수료율 개선안 제시

- 학습자 행동 로그를 기반으로 다양한 지표를 모니터링

    - 등록자는 많지만 실제 학습 시작자는 충분한가?
    - 시작한 학생들은 꾸준히 학습하는가?
    - 영상 시청, 챕터 탐색, 포럼 참여 같은 행동은 수료와 어떤 관련이 있는가?
    - 특정 코스나 특정 학습자 집단에서 이탈이 더 심한가?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import matplotlib.pyplot as plt
import platform
import os
import glob

plt.rcParams['font.family'] = 'Malgun Gothic'

In [2]:
df = pd.read_csv("data/Courses.csv")

# 데이터 확인

In [3]:
df.head()

,index,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,...,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
0,0,HarvardX/CB22x/2013_Spring,MHxPC130442623,1,0,0,0,United States,NaN,NaN,...,0,2012-12-19,2013-11-17,NaN,9.0,NaN,NaN,0,NaN,1.0
1,1,HarvardX/CS50x/2012,MHxPC130442623,1,1,0,0,United States,NaN,NaN,...,0,2012-10-15,NaN,NaN,9.0,NaN,1.0,0,NaN,1.0
2,2,HarvardX/CB22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,...,0,2013-02-08,2013-11-17,NaN,16.0,NaN,NaN,0,NaN,1.0
3,3,HarvardX/CS50x/2012,MHxPC130275857,1,0,0,0,United States,NaN,NaN,...,0,2012-09-17,NaN,NaN,16.0,NaN,NaN,0,NaN,1.0
4,4,HarvardX/ER22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,...,0,2012-12-19,NaN,NaN,16.0,NaN,NaN,0,NaN,1.0


In [4]:
df.shape

(641138, 21)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 641138 entries, 0 to 641137
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   index              641138 non-null  int64  
 1   course_id          641138 non-null  str    
 2   userid_DI          641138 non-null  str    
 3   registered         641138 non-null  int64  
 4   viewed             641138 non-null  int64  
 5   explored           641138 non-null  int64  
 6   certified          641138 non-null  int64  
 7   final_cc_cname_DI  641138 non-null  str    
 8   LoE_DI             535130 non-null  str    
 9   YoB                544533 non-null  float64
 10  gender             554332 non-null  str    
 11  grade              592766 non-null  str    
 12  start_time_DI      641138 non-null  str    
 13  last_event_DI      462184 non-null  str    
 14  nevents            441987 non-null  float64
 15  ndays_act          478395 non-null  float64
 16  nplay_video  

In [6]:
df.isna().sum()

index                     0
course_id                 0
userid_DI                 0
registered                0
viewed                    0
explored                  0
certified                 0
final_cc_cname_DI         0
LoE_DI               106008
YoB                   96605
gender                86806
grade                 48372
start_time_DI             0
last_event_DI        178954
nevents              199151
ndays_act            162743
nplay_video          457530
nchapters            258753
nforum_posts              0
roles                641138
incomplete_flag      540977
dtype: int64

# 주제 확인

### 중요 컬럼명
- **registered**: 강의 등록 여부
- **viewed**: 강의를 실제로 열어본 적이 있는지
- **explored**: 강의 콘텐츠를 더 깊게 탐색했는지
- **certified**: 최종적으로 인증서를 획득했는지
- **grade**: 학습 성취 수준
- **nevents**: 전체 학습 이벤트 수
- **ndays_act**: 실제 활동한 일수
- **nplay_video**: 영상 재생 수
- **nchapters**: 탐색한 챕터 수
- **nforum_posts**: 포럼 활동량


-> 학습자의 행동을 다음과 같은 퍼널로 해석할 수 있다

    - 등록 → 첫 방문 → 적극적 탐색 → 지속 학습 → 인증/수료

### 핵심 질문
- 우리 플랫폼의 전체 학습 퍼널은 어떻게 생겼는가?
- 학생들은 **어느 단계에서 가장 많이 이탈**하는가?
- **참여도가 높은 학생과 낮은 학생**은 어떤 행동 차이를 보이는가?
- 어떤 코스가 상대적으로 **참여 유지율 / 인증 전환율**이 높은가?
- 국가, 성별, 학력, 연령대 같은 특성에 따라 차이가 보이는가?
- 영상 재생 수, 활동일수, 챕터 탐색 수, 포럼 활동은 성취도나 인증과 어떤 관련이 있는가?
- 우리 서비스가 학생참여도를 높이기 위해 우선적으로 개선해야 할 지점은 무엇인가?

### 대시보드 작성 프로세스
- 분석 기획 완료 후 분석 주제 세분화
    - 학습자 관점 / 코스 관점 / 퍼널 관점
- 각 주제별로 어떤 질문에 답할지 정의
- 노트에 먼저 손으로 스케치
    - KPI 카드 위치
    - 필터 위치
    - 차트 간 연결 흐름
    - 사용자가 어떤 순서로 읽을지
- 스케치 후 워크시트 제작을 진행
- 제작 중 가시성이 떨어지거나 메시지가 약하면 과감히 수정

# 데이터 딕셔너리 

In [7]:
display(df.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
index,641138.0,NaN,NaN,NaN,320568.5,185080.742781,0.0,160284.25,320568.5,480852.75,641137.0
course_id,641138,16,HarvardX/CS50x/2012,169621,NaN,NaN,NaN,NaN,NaN,NaN,NaN
userid_DI,641138,476532,MHxPC130505428,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
registered,641138.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
viewed,641138.0,NaN,NaN,NaN,0.624299,0.484304,0.0,0.0,1.0,1.0,1.0
explored,641138.0,NaN,NaN,NaN,0.061899,0.240973,0.0,0.0,0.0,0.0,1.0
certified,641138.0,NaN,NaN,NaN,0.027587,0.163786,0.0,0.0,0.0,0.0,1.0
final_cc_cname_DI,641138,34,United States,184240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LoE_DI,535130,5,Bachelor's,219768,NaN,NaN,NaN,NaN,NaN,NaN,NaN
YoB,544533.0,NaN,NaN,NaN,1985.253279,8.891814,1931.0,1982.0,1988.0,1991.0,2013.0


In [10]:
df[['index']].value_counts()

index 
0         1
1         1
2         1
3         1
4         1
         ..
641133    1
641134    1
641135    1
641136    1
641137    1
Name: count, Length: 641138, dtype: int64

In [34]:
print(df['incomplete_flag'].dtypes)
print(df['incomplete_flag'].isnull().sum())
print((df['incomplete_flag'].isnull().mean() * 100).round(2))
print((df['incomplete_flag'].isnull().mean() * 100))

float64
540977
84.38
84.37762229036494


### 